# What This Walkthrough Does

Some websites do not put the visible data into the first HTML response. They
load it later with JavaScript.

This walkthrough compares:

- `requests`: fetches the initial server HTML
- Selenium: opens a real browser and observes the rendered DOM

# Imports and URL

## Packages Used in This Notebook

This walkthrough uses:

- `requests` to show what the server returns before JavaScript runs
- `Selenium` inside the browser function to open a real browser
- `pandas` to turn browser-extracted records into a table
- `Path` to save rendered HTML, screenshots, and CSV files
- `time` to pause after scrolling

The teaching contrast is: `requests` sees the initial HTML; Selenium sees the rendered browser DOM.


In [ ]:
# time lets us pause after browser actions such as scrolling.
import time
# Path helps create output folders and file paths.
from pathlib import Path

# pandas turns extracted browser records into tables.
import pandas as pd
# requests fetches the initial server HTML without running JavaScript.
import requests


In [ ]:
URL = "https://quotes.toscrape.com/js/"
USER_AGENT = "methodsNET-VLOP-course/1.0 dynamic walkthrough"


# Fetch with requests First

`requests` does not execute JavaScript.

In [ ]:
# requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
static_response = requests.get(URL, headers={"User-Agent": USER_AGENT}, timeout=30)
# raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
static_response.raise_for_status()

static_html = static_response.text

print("Static HTML characters:", len(static_html))
print("Does static HTML contain quote text?", "The world as we have created it" in static_html)


If this prints `False`, the visible quotes are not in the initial HTML returned
to `requests`.

# Define a Selenium Collection Function

Selenium controls a browser from Python.

In [ ]:
def collect_rendered_page_with_selenium(url: str, headless: bool = True):
    """Open a page in Chrome, wait for rendered content, and extract quote rows."""

    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait

    options = Options()
    options.add_argument(f"--user-agent={USER_AGENT}")

    if headless:
        options.add_argument("--headless=new")

    driver = webdriver.Chrome(options=options)

    try:
        # .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
        driver.get(url)

        wait = WebDriverWait(driver, 15)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".quote")))

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        # time.sleep(...) pauses between requests so the script does not make rapid-fire calls to a server.
        time.sleep(1)

        html = driver.page_source

        quote_blocks = driver.find_elements(By.CSS_SELECTOR, ".quote")
        quotes = []

        for block in quote_blocks:
            quote_text = block.find_element(By.CSS_SELECTOR, ".text").text
            author = block.find_element(By.CSS_SELECTOR, ".author").text
            tags = [
                tag.text
                for tag in block.find_elements(By.CSS_SELECTOR, ".tag")
            ]

            quotes.append(
                {
                    "quote": quote_text,
                    "author": author,
                    "tags": tags,
                }
            )

        screenshot = driver.get_screenshot_as_png()

    finally:
        driver.quit()

    return html, quotes, screenshot


::: {.callout-important}
Waiting is part of the method. If the wait condition is wrong, the scraper may
collect an empty or incomplete page state.
:::

# Run the Selenium Collection

Set `headless=False` during live teaching if you want students to watch the
browser window open.

In [ ]:
rendered_html, quotes, screenshot = collect_rendered_page_with_selenium(
    URL,
    headless=True,
)

print("Rendered HTML characters:", len(rendered_html))
print("Quotes extracted:", len(quotes))
# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
pd.DataFrame(quotes).head()


# Save Evidence

Rendered HTML and screenshots are raw evidence. A CSV is processed data.

In [ ]:
outdir = Path("data")
raw_dir = outdir / "raw"
processed_dir = outdir / "processed"

# mkdir(..., parents=True, exist_ok=True) creates the folder and any missing parent folders, without failing if it already exists.
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

rendered_path = raw_dir / "notebook_rendered_quotes_selenium.html"
screenshot_path = raw_dir / "notebook_rendered_quotes_selenium.png"
quotes_path = processed_dir / "notebook_rendered_quotes_selenium.csv"

rendered_path.write_text(rendered_html, encoding="utf-8")
screenshot_path.write_bytes(screenshot)
# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
pd.DataFrame(quotes).to_csv(quotes_path, index=False)

print(rendered_path)
print(screenshot_path)
print(quotes_path)


# Where Playwright Fits

Selenium is a classic browser automation tool and a good starting point.

Playwright is a newer alternative with strong auto-waiting, browser contexts,
network inspection, and tracing.

In this course:

- start with Selenium to understand browser automation
- mention Playwright as a modern alternative
- use Playwright later if network inspection or tracing becomes useful

# Exercise

Answer:

- What did `requests` miss?
- What did Selenium see?
- What wait condition did we use?
- What raw evidence did we save?
- When would browser automation be justified instead of an API?

# Key Takeaway

Browser automation observes a rendered browser state. That can be necessary,
but it also introduces new choices about waiting, scrolling, screenshots,
browser settings, and compliance.